# Lab 2: API and LLM Integration

Welcome to Lab 2 of Programming Methods! In this class we will be exploring APIs and LLMs. The Lab has **four** main exercises.

After cloning the repository (done in Lab 1) create a new branch for this Lab (e.g., `git checkout -b lab-02-apis`).

Then, complete the following exercises in a **new** Jupyter Notebook. 

Once you are done **commit your work** using the proper Git commands.

Good luck :) 


## A quick note before you start...

APIs let software talk to other software - most apps you use (maps,
weather, payments, social media) are quietly making API calls behind
the scenes. Learning to work with APIs means your code can pull in
live data instead of being limited to what you type by hand.

In this class, the **World Bank API** shows how to fetch open data,
and the **Wikipedia API** shows how to fetch and reuse text/content.
You will then connect both to **Ollama**, a tool for running AI models
*locally*. Once you understand REST APIs, "using AI" is just another API call.

This combination of public data + local AI reflects how a lot of
real software is built: gather data from external sources, then
compute or reason over it.

#### By the end of this class, you should be able to:

- Send GET and POST requests with the `requests` library.
- Pass parameters via query strings, URL paths, and JSON bodies.
- Check HTTP status codes explicitly instead of assuming success.
- Parse nested JSON to extract specific values.
- Explain, at a high level, how API keys/tokens authenticate requests.
- Chain multiple API calls together, using one's output as another's input.

## Setup

Run this cell first. If any of the packages is not installed run `uv add <package-name>`.

In [ ]:

import requests
import ollama

## What is a REST API?

A **REST API (Representational State Transfer API)** is a way for applications to communicate with each other over the web using **HTTP**.

A REST API exposes **resources** through URLs called **endpoints**. A client sends an HTTP request to an endpoint, and the server processes the request and returns an HTTP response.

For example, the following Wikipedia endpoint represents information about Portugal:

`https://en.wikipedia.org/api/rest_v1/page/summary/Portugal`

A typical interaction looks like:

**Client → HTTP Request → REST API → HTTP Response → Client**

### HTTP Methods

The HTTP method (or verb) indicates what operation the client wants to perform.

| HTTP Method | Purpose | Example |
|---|---|---|
| **GET** | Retrieve data | Get information about a country |
| **POST** | Send data / create a resource | Send a prompt to an LLM |
| **PUT** | Replace an existing resource | Replace a user profile |
| **PATCH** | Partially update a resource | Update a user's email |
| **DELETE** | Remove a resource | Delete a record |

### Requests

A request can contain information in different places, including:

- **URL path** — identifies the resource.
- **Query parameters** — provide additional options or filters.
- **Headers** — provide information about the request or client.
- **Body** — contains data sent to the server, commonly used with `POST`, `PUT`, and `PATCH`.

For example:

```python
response = requests.get(
    "https://api.worldbank.org/v2/country/pt/indicator/SP.POP.TOTL",
    params={"format": "json", "date": "2022"}
)

---
## Exercise 1: Public REST APIs (World Bank)

1. Choose a country and use the `requests` library to retrieve its **total population** (`SP.POP.TOTL`) from the World Bank API:

   `https://api.worldbank.org/v2/country/{country_code}/indicator/SP.POP.TOTL`

   Pass the following **query parameters** using the `params` argument of `requests.get()`:

   - `format`: request the response in JSON format.

   - `date`: choose the year for which you want to retrieve the population.

2. Explicitly check the HTTP status code returned by the API.

3. If the request is successful:

   - Parse the JSON response.

   - Extract the **country name**, **year**, and **population**.

   - Print these values.

In [3]:
# Write your World Bank API request here.
import requests

url = "https://api.worldbank.org/v2/country/pt/indicator/SP.POP.TOTL"
params = {"format": "json", "date": "2022"}

response = requests.get(url, params=params)

if response.status_code == 200:
    data = response.json()

    country = data[1][0]["country"]["value"]
    year = data[1][0]["date"]
    population = data[1][0]["value"]

    print(f"The country {country} in the year {year} had a population of {population}.")
else:
    print(f"Failed to retrieve data. HTTP Status Code: {response.status_code}")

The country Portugal in the year 2022 had a population of 10434332.


4. Make a second GET request to retrieve the metadata for the population indicator using:

   `https://api.worldbank.org/v2/indicator/SP.POP.TOTL`

   Again, request the response in JSON format using a query parameter.
   

5. If the request is successful:

   - Parse the JSON response.

   - Extract the indicator's **name**, **source**, and **description**.

   - Print these values.

In [4]:
# Write your indicator metadata request here.
url = "https://api.worldbank.org/v2/indicator/SP.POP.TOTL"
params = {"format": "json"}

response = requests.get(url, params=params)

if response.status_code == 200:
    data = response.json()

    name = data[1][0]["name"]
    source = data[1][0]["source"]["value"]
    description = data[1][0]["sourceNote"]

    print(name)
    print(source)
    print(description)
else:
    print(f"Failed to retrieve data. HTTP Status Code: {response.status_code}")

Population, total
World Development Indicators
Total population is based on the de facto definition of population, which counts all residents regardless of legal status or citizenship. The values shown are midyear estimates.


---
## Exercise 2: Public REST APIs (Wikipedia)

1. Create a list containing the names of **10 countries**.

2. Using a `for` loop, iterate over the countries and use the `requests` library to fetch the summary of each country from the Wikipedia REST API:

   `https://en.wikipedia.org/api/rest_v1/page/summary/{title}`

   Pass the country name as the article title in the URL path.

3. For each request, explicitly check whether the HTTP status code is `200`.

4. If the request is successful:

   - Parse the JSON response.

   - Extract the article's `title`, `extract` (summary), and page URL.

   - Print the extracted information.

5. If the request is not successful, print the country name and the HTTP status code returned by the API.

In [5]:
# Write your Wikipedia API loop here.
countries = ["Portugal", "Spain", "France", "Germany", "Hungary", "Italy", "Canada", "Australia", "Netherlands", "Andorra"]

headers = {
    "User-Agent": "programming-methods-lab02"
}

for country in countries:

    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{country}"

    response = requests.get(url, headers=headers)

    if response.status_code == 200:

        data = response.json()

        page_title = data["title"]
        page__extract = data["extract"]
        page_url = data["content_urls"]["desktop"]["page"]

        print(f"The page title is: {page_title}, the extract is: {page__extract}, and the page URL is: {page_url}.")

    else:
        print(f"Failed to retrieve data for {country}. HTTP Status Code: {response.status_code}")


The page title is: Portugal, the extract is: Portugal, officially the Portuguese Republic, is a country in Southwestern Europe. Mainland Portugal is located on the southwestern portion of the Iberian Peninsula, and bordered by Spain to the north and east. Portugal also includes the archipelagos of Madeira and the Azores in the Atlantic Ocean. The country has a population of 11.4 million, and Lisbon, its capital, is the largest city. Portugal's internal waters and territorial sea together account for two-fifths of its territory, and its exclusive economic zone is one of Europe's largest. The country's terrain contains a diverse range of landscapes and regional climates., and the page URL is: https://en.wikipedia.org/wiki/Portugal.
The page title is: Spain, the extract is: Spain, officially the Kingdom of Spain, is a country in Southern and Western Europe with territories in North Africa. Featuring the southernmost point of continental Europe, it is the largest country in Southern Europe

## Exercise 3: Interacting with Ollama's Local API

Let's connect the World Bank data to our local AI! 

1. Send a **POST** request to Ollama's local API (`http://localhost:11434/api/generate`) using the `requests` library.

2. Create a dictionary payload targeting the `llama3.2:1b` model (or the model you have installed).

3. For the `prompt`, use an f-string to inject the country and population variables you extracted in Exercise 1.

    - *Example: "Act as a grumpy wizard and explain why {country} has exactly {population} people."*

4. Set `stream`: False.

5. Write an if statement to check if the status code is 200. 

6. If the status code is 200, parse the JSON and print only the model's text response.

In [6]:
# Write your Ollama API request here.
import requests

prompt = f"Act as a grumpy wizard and explain why {country} has exactly {population} people."

payload = {
    "model": "llama3.2:1b",
    "prompt": prompt,
    "stream": False # response is sent as its produced (row by row) or the response is given at the end
}

url = "http://localhost:11434/api/generate"

response = requests.post(
    url,
    json=payload
)

if response.status_code == 200:

    data = response.json()

    print(data["response"])

else:
    print(f"failed. HTTP Status Code: {response.status_code}")

(Deep breath) Fine. I shall indulge your curious question, but don't come crying to me when I'm not providing you with any actual, verifiable information. (Muttering to himself) Andorra, indeed... 10434332... I suppose that's a perfectly acceptable number... for a country the size of Andorra... (Sigh)

Now, pay attention, pupil, as I reveal to you the reason behind this... (pausing to stroke his long white beard) ... astonishing number.

Andorra, a tiny principality nestled in the Pyrenees mountains, has a population that defies explanation. (Shaking his head) You see, the official language is Catalan, but I'm sure you're not interested in that. No, no, you want to know why there are exactly 10434332 people. Well, it's quite simple, really.

You see, Andorra is a microstate, with a total area of approximately 470 square kilometers. (Skeptical tone) Yes, that's a small country, even for a dwarf like myself. But, I suppose that's why they can afford to maintain their peculiar system of g

7. Send a second request to Ollama: use the Wikipedia extract text from Exercise 2 as the basis for a new prompt (e.g. ask the model to summarize it in one sentence, rewrite it in a different tone, or explain it "like I'm five"). Print the response.

In [7]:
# Use the Wikipedia extract from Exercise 2 in a second Ollama request.
extract_trimmed = page__extract[:500]

wiki_payload = {
    "model": "llama3.2:1b",
    "prompt": f"Sumarize the following extract from Wikipedia in a grumy wizard style: {extract_trimmed}.",
    "stream": False
}

wiki_response = requests.post(url, json=wiki_payload)

if wiki_response.status_code == 200:
    wiki_data = wiki_response.json()
    print(wiki_data["response"])
else:
    print(f"Failed to retrieve data from Ollama API. HTTP Status Code: {wiki_response.status_code}")

(Grumby wizard voice) Ah, mortal, gather 'round and listen closely, for I shall regale thee with tales of the land of Andorra. (puffs on a pipe)

In the distant lands of the Iberian Peninsula, where the Pyrenees Mountains pierce the sky like giant's fangs, there lies a land of shadows and secrecy. Andorra, a mystical realm of old, hath been shrouded in mystery since the dawn of time. (waves hand in a circular motion)

This cursed place, nestled between the French and Spanish lands, hath been home to the mighty Count of Urgell since the days of old. (scowls) But, alack! The Count's dominion did not last forever, for in 988, the treacherous French didst claim it as their own. (spits on the ground) Ah, the very thought doth make me shudder!

And then, as if the gods themselves didst intervene, the Count didst transfer the realm to the diocese of Urgell, a pitiful excuse for a church, if thou wilt. (scoffs) And thus, Andorra was formed, a polity born of a charter, a weak and feeble attempt

## A Note on Authenticated APIs

The World Bank and Wikipedia APIs don't require
authentication. Many other APIs (e.g. OpenWeatherMap, NASA API, Twitter/X,
GitHub) require an **API key or token** to identify who's making requests,
track usage, and enforce rate limits.

Typically, the key is sent either:
- As a query parameter: `params = {"appid": "YOUR_API_KEY", ...}`
- As a header: `headers = {"Authorization": "Bearer YOUR_TOKEN"}`

You can see an example using NASA's APOD (Astronomy Picture of the Day) API in the cell below.

**⚠️ Keys should never be hardcoded in shared code or notebooks, they're
often loaded from environment variables (`os.environ`) or a `.env` file.**

In [8]:
url = "https://api.nasa.gov/planetary/apod"
params = {"api_key": "DEMO_KEY"}  # rate-limited demo key, no signup needed

response = requests.get(url, params=params)

if response.status_code == 200:
    data = response.json()
    print(data['title'])
    print(data['explanation'])
else:
    print(f"Failed to retrieve data. HTTP Status Code: {response.status_code}")

A Treasure Chest in the Carina Nebula
This treasure chest is full of stars.   The featured image was obtained with NASA's James Webb Space Telescope and shows a dust pilar in the Carina Nebula inside our Galaxy, roughly 7500 light-years away.   It is formed by interstellar gas and dust, and shaped by powerful stellar winds and radiation form neighboring stars like the nearby Eta Carinae stellar system which is more luminous than 5 million suns.   The star formation inside the pillar is excavating its head, creating the open lid of the chest.     Astronomers estimate that there are about 70 stars in a compact cluster inside the pillar.   This cluster is now thought to be only around 1.3 million years old.   Its bounty of young stars includes a massive star approximately 19 times as massive as the Sun.   More massive stars are rarer, shine brighter and evolve faster than less massive stars.   They are the shiniest jewels in the treasure chest.     APOD's main NASA site is moving : From a

## Using the Ollama Python Library

So far, we have interacted with Ollama directly through its HTTP API using `requests`.

Ollama also provides a Python library that simplifies this interaction. Instead of manually creating and sending HTTP requests, we can use functions such as `ollama.chat()`.

In [15]:
response = ollama.chat(
    model="llama3.2:1b",
    messages=[
        {
            "role": "user",
            "content": "What is the capital of Portugal?"
        }
    ]
)

print(response["message"]["content"])

The capital of Portugal is Lisbon.


## Ollama Conversation History

If we want to build a chatbot that can follow a conversation, we need to provide the model with the **conversation history**.

Ollama's `chat()` function accepts a list of messages. Each message contains:

- `role`: who sent the message (`user`, `assistant`, or `system`)
- `content`: the text of the message

For example:

```python
messages = [
    {"role": "user", "content": "What is the capital of Portugal?"},
    {"role": "assistant", "content": "The capital of Portugal is Lisbon."},
    {"role": "user", "content": "What river runs through it?"}
]

For a practical use case check the examples below.

In [16]:
messages = []

user_input = "What is the capital of Portugal?"

messages.append({
    "role": "user",
    "content": user_input
})

response = ollama.chat(
    model="llama3.2:1b",
    messages=messages
)

bot_reply = response["message"]["content"]

messages.append({
    "role": "assistant",
    "content": bot_reply
})

print(bot_reply)

The capital of Portugal is Lisbon.


In [17]:
user_input = "And what is its population?"

messages.append({
    "role": "user",
    "content": user_input
})

response = ollama.chat(
    model="llama3.2:1b",
    messages=messages
)

bot_reply = response["message"]["content"]

messages.append({
    "role": "assistant",
    "content": bot_reply
})

print(bot_reply)

As of my cut-off knowledge date of December 2023, the population of Lisbon, the capital city of Portugal, is approximately 505,000 people within the city limits, but the metropolitan area has a population of around 2.8 million, making it the second-largest metropolitan area in Portugal.


## Exercise 4: Build a CLI Chatbot

Now that we know how to send messages to Ollama and maintain conversation history, let's build a simple command-line chatbot.

Create a new file called `chatbot.py`.

Your chatbot should:

1. Create an empty `messages` list to store the conversation history.

2. Continuously ask the user for input using a `while` loop.

3. If the user enters `exit` or `quit`, terminate the program.

4. Add each user message to the conversation history using the following format:

   `{"role": "user", "content": user_input}`

5. Send the complete conversation history to `ollama.chat()` using the `llama3.2:1b` model (or the model you have installed).

6. Extract and print the assistant's response.

7. Add the assistant's response to the conversation history so that it is available in the next interaction.

Run your chatbot from the terminal (in the directory where the file is stored):

`python chatbot.py`

Try asking a follow-up question that depends on something you said earlier. Does the chatbot remember the context?

In [18]:
# Create chatbot.py as described in Exercise 4.
messages = []

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Goodbye!")
        break

    messages.append({
        "role": "user",
        "content": user_input
    })

    response = ollama.chat(
        model="llama3.2:1b",
        messages=messages
    )

    bot_reply = response["message"]["content"]

    print("Bot:", bot_reply)

    messages.append({
        "role": "assistant",
        "content": bot_reply
    })

KeyboardInterrupt: Interrupted by user

## Wrap-up Questions

### 1. Why do we use `GET` with query parameters for the World Bank API, but `POST` with a JSON payload for the Ollama API?

A. Because `GET` only works with public APIs, while `POST` is required for APIs running locally.

X B. Because `GET` is typically used to retrieve existing resources, while `POST` can be used to send input to the server for processing.

C. Because query parameters can only be used with `GET`, and JSON can only be used with `POST`.

D. Because `POST` requests are always more secure than `GET` requests.

---

### 2. Why should we check the HTTP status code before processing the response?

A. To make the API return JSON instead of plain text.

B. To make the request execute faster.

X C. To verify that the request succeeded before assuming the response contains the expected data.

D. To prevent the API from receiving too many requests.

---

### 3. What does `"stream": False` change when making a request to Ollama?

A. It prevents Ollama from accessing the internet.

B. It makes Ollama generate a shorter response.

X C. It tells Ollama to return the complete generated response as a single response instead of sending it progressively in chunks.

D. It prevents the model from remembering previous messages.

---

### 4. Why do we send the complete `messages` history when calling `ollama.chat()`?

X A. Because the model does not automatically remember previous API calls, so previous messages must be provided as context.

B. Because Ollama requires at least two messages before it can generate a response.

C. Because storing messages makes the model generate responses faster.

D. Because the `messages` list is automatically stored permanently by Ollama.


## Optional Stretch Goal: PokeAPI

Follow the same reasoning as in exercise 1 but in this case use the **PokeAPI**.

In [19]:
# Choose a Pokemon and query the PokeAPI.
pokemon_name = "snorlax"

import requests

pokemon_name = "snorlax"

url = f"https://pokeapi.co/api/v2/pokemon/{pokemon_name}"

response = requests.get(url)

if response.status_code == 200:
    data = response.json()

    name = data["name"]
    pokemon_id = data["id"]
    height = data["height"]
    weight = data["weight"]

    print("Name:", name)
    print("ID:", pokemon_id)
    print("Height:", height)
    print("Weight:", weight)
else:
    print(f"Failed to retrieve data. HTTP Status Code: {response.status_code}")

Name: snorlax
ID: 143
Height: 21
Weight: 4600


## 